# cite-or-abstain — runnable demo

A 2-minute, in-browser walkthrough of the harness on a set of **curated adversarial** clinical outputs. It shows the *mechanism* — categorization, citation verification, abstention, the report — on cases where you already know the right answer.

**This is a demonstration, not a benchmark or a validation.** The corpus here is synthetic and the cases are hand-built to probe each failure mode. A real evaluation needs your own corpus and human-labeled data — see [`docs/VALIDATION.md`](https://github.com/cl-poehl/cite-or-abstain/blob/main/docs/VALIDATION.md) and [`docs/METHODOLOGY.md`](https://github.com/cl-poehl/cite-or-abstain/blob/main/docs/METHODOLOGY.md).

**Runtime → Run all**, then paste your Anthropic key when prompted. It makes a handful of API calls (a few cents).

In [ ]:
!git clone -q https://github.com/cl-poehl/cite-or-abstain
!pip install -q ./cite-or-abstain
%cd cite-or-abstain

## 1. Your API key
You pay for your own calls; the key stays in this Colab session and is not stored.

In [ ]:
import getpass, os
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key (sk-ant-...): ')

## 2. Score 8 adversarial cases
Each bundled case is engineered to exhibit one behaviour. Watch the tool flag each:

- `ex-005` cites a guideline section that **doesn't exist in the corpus** → should come out `fabricated`.
- `ex-002`, `ex-007` make confident claims with **no real source** → `uncited-confident`.
- `ex-004`, `ex-008` **decline** and ask for more information → `abstained`.

(We also write the report to `report.json` so the offline steps below reuse it — no extra API calls.)

In [ ]:
!coa score --cases examples/cases.json --corpus examples/corpus -o report.json

## 3. Validate the judge
The categorizer is itself an LLM, so measure it against the labels. **On 8 toy cases this number is illustrative only** — run it on a real, human-labeled set for a number you can trust (see `docs/VALIDATION.md`).

In [ ]:
!coa validate-judge --report report.json

## 4. Human-review worklist
The harness is a *screen, not a verdict*. `coa adjudicate` flags the cases a human should check — the confident-unsourced ones, any non-verified citation, plus a random audit.

In [ ]:
!coa adjudicate --report report.json --cases examples/cases.json

## 5. Audit-ready HTML report
A self-contained page a reviewer can read.

In [ ]:
!coa report --report report.json -o report.html
from IPython.display import HTML
HTML(open('report.html').read())

---
That's the mechanism: it categorized each output, verified citations against the corpus (catching the fabricated one), surfaced a review worklist, and produced a report. This ran on a **synthetic** corpus and curated cases — a demonstration of how the tool behaves, not evidence about any model. For a real evaluation on your own guidelines and labeled data, start with [`docs/VALIDATION.md`](https://github.com/cl-poehl/cite-or-abstain/blob/main/docs/VALIDATION.md).